In [1]:
import json
import re
from pathlib import Path

Собираем объекты файлов в список

In [2]:
DATA_DIR = Path("../data/ru-hard-detection-dataset")
files = [f for f in DATA_DIR.rglob("*") if f.is_file() and not f.name.startswith("paraphrased")]
files

[WindowsPath('../data/ru-hard-detection-dataset/essay/generated_essays.json'),
 WindowsPath('../data/ru-hard-detection-dataset/essay/original_essay.json'),
 WindowsPath('../data/ru-hard-detection-dataset/news/generated_news.json'),
 WindowsPath('../data/ru-hard-detection-dataset/news/original_news.json'),
 WindowsPath('../data/ru-hard-detection-dataset/scientific_texts/generated_scientific.json'),
 WindowsPath('../data/ru-hard-detection-dataset/scientific_texts/orig_scientific.json')]

In [3]:
data = []
for file in files:
    with open(file, 'r', encoding="utf-8") as f:
        data.extend(json.load(f))
len(data)

2878

In [4]:
data[0]

{'id': 1,
 'text': '### Советский Союз 1922–1939: Формирование, Реформы и Их Последствия\n\n#### Введение\n\nПериод с 1922 по 1939 год стал одним из наиболее трансформационных этапов в истории Советского Союза. Основание СССР, приход к власти Иосифа Виссарионовича Сталина, реализация масштабных экономических и социальных реформ, а также активная внешняя политика определили дальнейшее развитие страны. Этот период характеризуется как выдающимися достижениями в области индустриализации и науки, так и трагическими последствиями для миллионов граждан. Данное эссе рассматривает ключевые события и процессы, происходившие в СССР в указанный период, анализируя их влияние на внутреннюю структуру государства и его международное положение.\n\n#### Образование Советского Союза и Международное Признание\n\n14 декабря 1922 года состоялось официальное образование Союза Советских Социалистических Республик (СССР), объединившего Российскую, Украинскую, Белорусскую и Закавказскую Советские Социалистическ

Разделяем человеческие и ai тексты

In [5]:
ai_data = [e for e in data if "ai" in e["source"]]
human_data = [e for e in data if e["source"] != "ai"]

In [6]:
def clean_ai_text(text: str):
    text = re.sub(r'[*_]{1,3}', '', text)
    text = re.sub(r'#+?', '', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


clean_data = []
for item in data:
    if "ai" in item["source"]:
        item["text"] = clean_ai_text(item["text"]) 
        clean_data.append(item)
    else:
        clean_data.append(item)
clean_data[0]

{'id': 1,
 'text': 'Советский Союз 1922–1939: Формирование, Реформы и Их Последствия Введение Период с 1922 по 1939 год стал одним из наиболее трансформационных этапов в истории Советского Союза. Основание СССР, приход к власти Иосифа Виссарионовича Сталина, реализация масштабных экономических и социальных реформ, а также активная внешняя политика определили дальнейшее развитие страны. Этот период характеризуется как выдающимися достижениями в области индустриализации и науки, так и трагическими последствиями для миллионов граждан. Данное эссе рассматривает ключевые события и процессы, происходившие в СССР в указанный период, анализируя их влияние на внутреннюю структуру государства и его международное положение. Образование Советского Союза и Международное Признание 14 декабря 1922 года состоялось официальное образование Союза Советских Социалистических Республик (СССР), объединившего Российскую, Украинскую, Белорусскую и Закавказскую Советские Социалистические Республики. Этот шаг бы

Средняя длина текста по словам

In [7]:
word_lens = list(map(lambda x: len(x['text'].split()), clean_data))
round(sum(word_lens) / len(word_lens), 4)

854.7787

Инициализируем модель для чанкинга

In [8]:
from transformers import BertConfig, AutoTokenizer, AutoModel

MODEL_NAME = "DeepPavlov/rubert-base-cased-sentence"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)
config = BertConfig.from_pretrained(
    MODEL_NAME
)
model = AutoModel.from_pretrained(
    MODEL_NAME
)

print(config.max_position_embeddings)
print(model)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

512
BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=Fals

Функция чанкования текста

In [9]:
def chunk_text(text: str, chunk_size: int = 500):
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    for i in range(0, len(tokens), chunk_size):
        chunk = tokens[i:i + chunk_size]
        
        if len(chunk) < 50:
            continue

        chunks.append(tokenizer.decode(chunk))
    return chunks

### Лингвистический анализ

In [16]:
from natasha import Segmenter, NewsEmbedding, NewsMorphTagger, NewsSyntaxParser, Doc

In [17]:
segmenter = Segmenter()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
syntax_parser = NewsSyntaxParser(emb)

In [18]:
def get_chunk_linguistics(text):
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    doc.parse_syntax(syntax_parser)
    
    tokens = doc.tokens
    total = len(tokens)
    if total == 0:
        return {"noun_ratio": 0, "verb_ratio": 0, "adj_ratio": 0, "avg_word_len": 0, "perf_verb_ratio": 0, "syntax_depth_avg": 0}
    
    # Считаем части речи
    pos_counts = {"NOUN": 0, "VERB": 0, "ADJ": 0, "PUNCT": 0, "PRON": 0}
    perf_verbs = 0 # Совершенный вид
    syntax_rels = 0 # Количество связей
    word_lengths = []
    
    for t in tokens:
        if t.pos in pos_counts:
            pos_counts[t.pos] += 1
        if t.pos != 'PUNCT':
            word_lengths.append(len(t.text))
        if t.pos == 'VERB' and t.feats and t.feats.get('Aspect') == 'Perf':
            perf_verbs += 1
        if t.head_id and t.head_id != '0':
            syntax_rels += 1
            
    return {
        "noun_ratio": pos_counts["NOUN"] / total,
        "verb_ratio": pos_counts["VERB"] / total,
        "adj_ratio": pos_counts["ADJ"] / total,
        "pron_ratio": pos_counts["PRON"] / total,
        "perf_verb_ratio": perf_verbs / (pos_counts["VERB"] + 1e-5), # Доля сов. вида
        "punct_ratio": pos_counts["PUNCT"] / total,
        "avg_syntax_links": syntax_rels / total, # сложность структуры
        "avg_word_len": sum(word_lengths) / len(word_lengths) if word_lengths else 0
    }

Чанкуем текста под длину берта И сохраняем случайный чанк

In [19]:
import random

for item in clean_data:
    chosen_chunk = random.choice(chunk_text(item["text"]))
    item["text"] = chosen_chunk

    features = get_chunk_linguistics(chosen_chunk)
    
    item.update(features)
print(len(clean_data))
print(clean_data[0])

2878
{'id': 1, 'text': 'условиями труда и ростом преступности. Коллективизация и Ее Последствия Одной из наиболее разрушительных реформ стал процесс коллективизации сельского хозяйства. Стремясь ликвидировать крестьянское хозяйство и создать коллективные фермы ( колхозы и совхозы ), советская власть столкнулась с ожесточенным сопротивлением крестьянства. Коллективизация сопровождалась насильственными конфискациями земли, скота и имущества, что привело к массовым репрессиям и голоду. Наиболее трагическими последствиями коллективизации стали массовый голод 1932 – 1933 годов, особенно в Украине ( Голодомор ), и масштабные репрессии против крестьян, обвиняемых в " антисоветской деятельности ". По оценкам историков, от голода и репрессий погибло несколько миллионов человек. Эти события оставили глубокий след в демографическом и социальном развитии страны, породив длительную травму в коллективной памяти нации. Конституции СССР 1924 и 1936 годов Принятие Конституции СССР 1924 года ознаменовал

Загружаем данные в конечный json-файл

In [20]:
import pandas as pd

OUTPUT_FILE = "ru-hard-detection-clean-v2.csv"
df = pd.DataFrame(clean_data)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")